# Exercise 6: Identify, then transliterate

*Scripts and Encoding · version 0.1.0*

> Choose 20 Kyrgyz sentences. Run `turkicnlp.LanguageDetection()` on them as they are, then again after converting them to the Common Turkic Alphabet. How does accuracy change, and which wrong labels appear? Relate the result to the language-identification section.

The notebook uses 20 Kyrgyz sentences from the X-WMT test set. `LanguageDetection()` downloads the GlotLID model from Hugging Face on first use and needs `fasttext`.

**Setup.** `pip install "turkicnlp==0.5.0"[lid] requests`. Cells that fetch data from the web need internet access.

In [ ]:
import requests

BASE = "https://raw.githubusercontent.com/turkic-interlingua/til-mt/master/xwmt/test"

def xwmt_side(pair, lang):
    """One side of an X-WMT test pair, e.g. xwmt_side("en-ky", "ky")."""
    r = requests.get(f"{BASE}/{pair}/{pair}.{lang}", timeout=30)
    r.raise_for_status()
    return r.text.splitlines()

In [ ]:
from collections import Counter
import turkicnlp
from turkicnlp.scripts import Script
from turkicnlp.scripts.transliterator import Transliterator

ky = xwmt_side("en-ky", "ky")[:20]
to_cts = Transliterator("kir", Script.CYRILLIC, Script.COMMON_TURKIC)
lid = turkicnlp.LanguageDetection()          # restricted to Turkic labels

def top_labels(sentences):
    return [lid.predict(s, k=1)[0][0] for s in sentences]

native = top_labels(ky)
converted = top_labels([to_cts.transliterate(s) for s in ky])
for name, labels in [("native Cyrillic", native), ("CTS", converted)]:
    acc = sum(l == "__label__kir_Cyrl" for l in labels) / len(labels)
    print(f"{name:<16} accuracy {acc:.0%}  labels: {Counter(labels).most_common()}")

## Worked solution

This experiment needs the GlotLID model, which could not be downloaded in the environment where these notebooks were prepared, so no output is recorded here; run the cells to obtain yours.

**What to expect, and why.** The TurkicNLP paper reports that GlotLID restricted to Turkic labels identifies Kyrgyz almost perfectly in its native Cyrillic orthography, but that accuracy across languages drops to 48.1% on script-converted text. The identifier relies on orthographic cues: Kyrgyz *ө*, *ү* and *ң*, and the absence of Kazakh-only letters such as *ә*, *і*, *қ*, *ғ*, *ұ*. Converting to CTS removes exactly these cues, and the converted text resembles no orthography the model was trained on. Wrong labels on CTS text are typically other Latin-script Turkic languages (Turkish, Uzbek, Crimean Tatar).

**Relation to the chapter.** This is lesson 1 of the language-identification section: identify the language on the original text first, and treat CTS as a derived view for downstream processing, never as input to the identifier.